# Day 4 — PySpark Practice Questions: Aggregations

| # | Difficulty | Topic |
|---|-----------|-------|
| 1 | 🟢 Easy    | Total revenue (quantity × unit_price) per region |
| 2 | 🟢 Easy    | Products with more than 50 total units sold |
| 3 | 🟡 Medium  | Per region: min price, max price, avg price, total qty |

In [5]:
import os
import sys
import shutil
from pathlib import Path

# Portable PySpark environment setup: detect current Python and Java automatically.
# Avoid hardcoded instructor or machine-specific paths.
python_exe = sys.executable
os.environ['PYSPARK_PYTHON'] = python_exe
os.environ['PYSPARK_DRIVER_PYTHON'] = python_exe

java_exe = shutil.which('java')
if java_exe:
    java_home = Path(java_exe).resolve().parent.parent
    os.environ['JAVA_HOME'] = str(java_home)
else:
    raise RuntimeError('Java not found on PATH. Install a JDK and add java.exe to PATH, or set JAVA_HOME manually.')

print('Environment set.')
print('JAVA_HOME =', os.environ.get('JAVA_HOME'))
print('PYSPARK_PYTHON =', os.environ.get('PYSPARK_PYTHON'))
print('PYSPARK_DRIVER_PYTHON =', os.environ.get('PYSPARK_DRIVER_PYTHON'))

Environment set.
JAVA_HOME = C:\Users\Friend\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.3.9-hotspot
PYSPARK_PYTHON = c:\Users\Friend\Downloads\python-pyspark-sql-sessions\myenv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON = c:\Users\Friend\Downloads\python-pyspark-sql-sessions\myenv\Scripts\python.exe


In [6]:
import os


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day4_PySpark_PQ') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# Shared dataset — same as SQL notebook
sales_data = [
    ('S001','P001','North','2024-01-05', 10, 29.99),
    ('S002','P002','South','2024-01-07',  5, 49.99),
    ('S003','P001','East', '2024-01-10', 20, 29.99),
    ('S004','P003','West', '2024-01-12', 15, 99.99),
    ('S005','P002','North','2024-01-15',  8, 49.99),
    ('S006','P001','South','2024-02-01', 30, 29.99),
    ('S007','P003','East', '2024-02-03',  2, 99.99),
    ('S008','P004','West', '2024-02-08', 25, 14.99),
    ('S009','P004','North','2024-02-11', 40, 14.99),
    ('S010','P002','East', '2024-02-14', 12, 49.99),
    ('S011','P001','West', '2024-03-01', 18, 29.99),
    ('S012','P003','South','2024-03-05',  7, 99.99),
    ('S013','P004','East', '2024-03-09', 22, 14.99),
    ('S014','P002','West', '2024-03-12',  3, 49.99),
    ('S015','P001','North','2024-03-20', 14, 29.99),
]

schema = StructType([
    StructField('sale_id',    StringType(),  False),
    StructField('product_id', StringType(),  True),
    StructField('region',     StringType(),  True),
    StructField('sale_date',  StringType(),  True),
    StructField('quantity',   IntegerType(), True),
    StructField('unit_price', DoubleType(),  True),
])

df_sales = spark.createDataFrame(sales_data, schema=schema)
df_sales.printSchema()
df_sales.show(5)
print('Total rows:', df_sales.count())

Spark version: 4.1.2
root
 |-- sale_id: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)

+-------+----------+------+----------+--------+----------+
|sale_id|product_id|region| sale_date|quantity|unit_price|
+-------+----------+------+----------+--------+----------+
|   S001|      P001| North|2024-01-05|      10|     29.99|
|   S002|      P002| South|2024-01-07|       5|     49.99|
|   S003|      P001|  East|2024-01-10|      20|     29.99|
|   S004|      P003|  West|2024-01-12|      15|     99.99|
|   S005|      P002| North|2024-01-15|       8|     49.99|
+-------+----------+------+----------+--------+----------+
only showing top 5 rows
Total rows: 15


---
# Question 1 🟢 — Total Revenue per Region

## Concept Warm-up

In [9]:
from pyspark.sql import functions as F
df_sales.withColumn(
    'revenue',F.col('quantity')*F.col('unit_price')
).select('sale_id','product_id','region','sale_date','quantity','unit_price','revenue').show(5);

+-------+----------+------+----------+--------+----------+------------------+
|sale_id|product_id|region| sale_date|quantity|unit_price|           revenue|
+-------+----------+------+----------+--------+----------+------------------+
|   S001|      P001| North|2024-01-05|      10|     29.99|             299.9|
|   S002|      P002| South|2024-01-07|       5|     49.99|249.95000000000002|
|   S003|      P001|  East|2024-01-10|      20|     29.99|             599.8|
|   S004|      P003|  West|2024-01-12|      15|     99.99|           1499.85|
|   S005|      P002| North|2024-01-15|       8|     49.99|            399.92|
+-------+----------+------+----------+--------+----------+------------------+
only showing top 5 rows


In [10]:
# Warm-up 1: withColumn to create a computed column
df_sales.withColumn(
    'revenue', F.col('quantity') * F.col('unit_price')
).select('sale_id', 'region', 'quantity', 'unit_price', 'revenue').show(5)

+-------+------+--------+----------+------------------+
|sale_id|region|quantity|unit_price|           revenue|
+-------+------+--------+----------+------------------+
|   S001| North|      10|     29.99|             299.9|
|   S002| South|       5|     49.99|249.95000000000002|
|   S003|  East|      20|     29.99|             599.8|
|   S004|  West|      15|     99.99|           1499.85|
|   S005| North|       8|     49.99|            399.92|
+-------+------+--------+----------+------------------+
only showing top 5 rows


In [11]:
(df_sales
 .withColumn('revenue',F.col('quantity')* F.col('unit_price'))
 .groupBy('region')
 .agg(F.sum('revenue').alias('total_evenue'))
 .show(5)
 )

+------+------------------+
|region|      total_evenue|
+------+------------------+
| North|           1719.28|
| South|           1849.58|
|  East|1729.4399999999998|
|  West|           2564.39|
+------+------------------+



In [12]:
# Warm-up 2: groupBy + F.sum on the revenue column
(
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region')
    .agg(F.sum('revenue').alias('total_revenue_raw'))
    .show()
)

+------+------------------+
|region| total_revenue_raw|
+------+------------------+
| North|           1719.28|
| South|           1849.58|
|  East|1729.4399999999998|
|  West|           2564.39|
+------+------------------+



In [ ]:
# Warm-up 3: F.round to round to 2 decimal places
(
    df_sales
    .withColumn('revenue', F.col('quantity') * F.col('unit_price'))
    .groupBy('region')
    .agg(F.round(F.sum('revenue'), 2).alias('total_revenue'))
    .show()
)

## Problem

Compute total revenue (`quantity × unit_price`) per region.  
Sort by `total_revenue` **descending** — highest revenue first.

**Expected columns:** `region`, `total_revenue`

In [20]:
# YOUR CODE HERE
df_revenue = df_sales.withColumn(
    'revenue',F.col('quantity') * F.col('unit_price')).groupBy(
        'region').agg(
            F.round(F.sum('revenue'),2
                    ).alias('total_revenue')).orderBy(
                'total_revenue',desc=True).select(
                    'region','total_revenue')

df_revenue.show(5)

+------+-------------+
|region|total_revenue|
+------+-------------+
| North|      1719.28|
|  East|      1729.44|
| South|      1849.58|
|  West|      2564.39|
+------+-------------+



In [ ]:
# YOUR CODE HERE
df_revenue = df_sales.withColumn(
    'revenue',F.col('quantity') * F.col('unit_price')).groupBy(
        'region').agg(
            F.round(F.sum('revenue'),2
                    ).alias('total_revenue')).orderBy(
                'revenue',desc=True).select(
                    'region','total_revenue')

df_revenue.show(5)

---
# Question 2 🟢 — Products with More Than 50 Units Sold

## Concept Warm-up

In [ ]:
# Warm-up 1: total units per product — no filter yet
(
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units'))
    .orderBy('total_units', ascending=False)
    .show()
)

In [ ]:
# Warm-up 2: .filter() after .agg() = SQL HAVING
# Filter threshold = 30 to see what the pattern looks like
(
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units'))
    .filter(F.col('total_units') > 30)        # ← THIS is HAVING
    .show()
)

In [ ]:
# Warm-up 3: orderBy descending
(
    df_sales
    .groupBy('product_id')
    .agg(F.sum('quantity').alias('total_units'))
    .orderBy('total_units', ascending=False)   # highest first
    .show()
)

## Problem

Find all products where total quantity sold **exceeds 50**.  
Sort by total units **descending**.

**Expected columns:** `product_id`, `total_units_sold`

**Hints:**
- `.groupBy('product_id').agg(F.sum('quantity').alias('total_units_sold'))`
- `.filter(F.col('total_units_sold') > 50)` — PySpark's HAVING
- `.orderBy('total_units_sold', ascending=False)`

In [21]:
# YOUR CODE HERE
df_top_products = df_sales.groupBy('product_id'
                                   ).agg(F.sum('quantity'
                                                           ).alias('total_units_sold')).filter(F.col('total_units_sold')>50).orderBy('total_units_sold',ascending=False)
df_top_products.show()

+----------+----------------+
|product_id|total_units_sold|
+----------+----------------+
|      P001|              92|
|      P004|              87|
+----------+----------------+



---
# Question 3 🟡 — Regional Price & Quantity Summary

## Concept Warm-up

In [ ]:
# Warm-up 1: F.min and F.max
(
    df_sales
    .groupBy('region')
    .agg(
        F.min('unit_price').alias('min_price'),
        F.max('unit_price').alias('max_price'),
    )
    .orderBy('region')
    .show()
)

In [ ]:
# Warm-up 2: F.avg and F.round
(
    df_sales
    .groupBy('region')
    .agg(
        F.round(F.avg('unit_price'), 2).alias('avg_price'),
    )
    .orderBy('region')
    .show()
)

In [ ]:
# Warm-up 3: multiple aggregates in one .agg() call — one shuffle pass
(
    df_sales
    .groupBy('region')
    .agg(
        F.count('*').alias('num_transactions'),
        F.sum('quantity').alias('total_qty'),
    )
    .orderBy('region')
    .show()
)

## Problem

For each region produce a summary with:
- `min_price` — minimum `unit_price`
- `max_price` — maximum `unit_price`
- `avg_price` — average `unit_price` rounded to 2dp
- `total_qty` — total `quantity` sold

Order by `region` alphabetically.

**Hints:** Put all four aggregates in a single `.agg()` call. Use `F.round(F.avg(...), 2)` for the average.

In [22]:
# YOUR CODE HERE
df_region_summary = df_sales.groupBy('region').agg(
    F.min('unit_price').alias('min_price'),
    F.max('unit_price').alias('max_price'),
    F.round(F.avg('unit_price'),2).alias('avg_price'),
    F.sum('quantity').alias('total_qty')
).orderBy('region',ascending=True)

df_region_summary.show()

+------+---------+---------+---------+---------+
|region|min_price|max_price|avg_price|total_qty|
+------+---------+---------+---------+---------+
|  East|    14.99|    99.99|    48.74|       56|
| North|    14.99|    49.99|    31.24|       72|
| South|    29.99|    99.99|    59.99|       42|
|  West|    14.99|    99.99|    48.74|       61|
+------+---------+---------+---------+---------+



In [23]:
spark.stop()
print('Spark stopped.')

Spark stopped.
